# F1 Strategy Predictor
## Real-Time Pit Stop & Compound Prediction (PyTorch)

---

This notebook trains two LSTM models for real-time F1 race strategy prediction:
1. **PIT MODEL**: Predicts if a driver will pit within the next 3 laps (binary)
2. **COMPOUND MODEL**: Predicts which tire compound will be used next (multiclass)

Architecture: LSTM + Multi-Head Attention + Dense layers

### Requirements
- `label_encoder.pkl` (from DataAnalysis.ipynb)
- `f1_dataset_clean.pkl` (from DataAnalysis.ipynb)

### Framework
**PyTorch 2.0+** with CUDA support (optional but recommended)

# 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

In [ ]:
import joblib

import json

import os
os.makedirs('Model', exist_ok=True)
os.makedirs('Other', exist_ok=True)

In [ ]:
import importlib.util
if importlib.util.find_spec('fastf1') is None:
    !pip install fastf1 --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# Check PyTorch and device
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (confusion_matrix, accuracy_score, f1_score, 
                             roc_auc_score, precision_recall_curve,
                             classification_report)

# 2. Load Dataset

In [ ]:
# Load preprocessed dataset
df_f1 = pd.read_pickle('f1_dataset_clean.pkl')
print(f"Dataset loaded: {len(df_f1):,} rows")

In [ ]:
# Load pre-fitted LabelEncoder
label_encoder = joblib.load('label_encoder.pkl')
print(f"Classes: {list(label_encoder.classes_)}")

# 3. Define Features

In [ ]:
# Define targets and metadata
TARGETS = ['PitIn3Laps', 'NextCompoundEnc']
META_COLS = ['Driver', 'Team', 'Circuit', 'Time']

FEATURES = [col for col in df_f1.columns if col not in TARGETS + META_COLS]

---
# 4. Data Preprocessing

### 4.1 Create Temporal Sequences

Convert 2D dataframe into 3D sequences for LSTM input.

In [ ]:
SEQUENCE_LENGTH = 10  # Number of past laps

In [ ]:
def create_sequences(df, target_col, features=FEATURES, seq_len=SEQUENCE_LENGTH):
    """
    Transform flat data into temporal sequences for LSTM.
    
    Returns:
        X: (n_samples, seq_len, n_features) - Input sequences
        y: (n_samples,) - Target values
        lengths: (n_samples,) - Valid sequence lengths (for masking)
    """
    X = []
    y = []
    lengths = []

    # Group by stint
    for (_, _, _, _), group in df.groupby(['Year', 'Round', 'Driver', 'Stint']):
        group = group.sort_values('LapNumber')

        if len(group) < 3:
            continue

        data = group[features].values.astype(np.float32)
        targets = group[target_col].values

        # Create sequences
        for i in range(1, len(group)):
            start = max(0, i - seq_len)
            seq = data[start:i]
            
            # Track actual length before padding
            actual_len = len(seq)
            
            # Pad if needed
            if len(seq) < seq_len:
                pad = np.zeros((seq_len - len(seq), len(features)), dtype=np.float32)
                seq = np.vstack([pad, seq])

            X.append(seq)
            y.append(targets[i])
            lengths.append(actual_len)

    return np.array(X), np.array(y), np.array(lengths)


def create_sequencesP(df, target_col='PitIn3Laps'):
    return create_sequences(df, target_col)

def create_sequencesC(df, target_col='NextCompoundEnc'):
    return create_sequences(df, target_col)

### 4.2 Train/Validation/Test Split

In [ ]:
# Temporal split
df_clean = df_f1.sort_values(['Year', 'Round', 'LapNumber']).reset_index(drop=True)
df_clean['RaceID'] = df_clean['Year'].astype(str) + '_' + df_clean['Round'].astype(str).str.zfill(2)

In [ ]:
# Split races (70% / 15% / 15%)
SPLIT_PERC = [0.70, 0.85]
races = sorted(df_clean['RaceID'].unique())
n = len(races)

df_train = df_clean[df_clean['RaceID'].isin(races[:int(SPLIT_PERC[0]*n)])].copy()
df_val = df_clean[df_clean['RaceID'].isin(races[int(SPLIT_PERC[0]*n):int(SPLIT_PERC[1]*n)])].copy()
df_test = df_clean[df_clean['RaceID'].isin(races[int(SPLIT_PERC[1]*n):])].copy()

print(f'Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}')

In [ ]:
# Create sequences
X_pit_train, y_pit_train, len_pit_train = create_sequencesP(df=df_train)
X_pit_val, y_pit_val, len_pit_val = create_sequencesP(df=df_val)
X_pit_test, y_pit_test, len_pit_test = create_sequencesP(df=df_test)

X_comp_train, y_comp_train, len_comp_train = create_sequencesC(df=df_train)
X_comp_val, y_comp_val, len_comp_val = create_sequencesC(df=df_val)
X_comp_test, y_comp_test, len_comp_test = create_sequencesC(df=df_test)

print(f'PIT: {X_pit_train.shape} | COMPOUND: {X_comp_train.shape}')

### 4.3 Normalization

In [ ]:
scaler_pit = RobustScaler()
scaler_comp = RobustScaler()

n_feat = X_pit_train.shape[2]

In [ ]:
# Reshape, scale, reshape back
X_pit_train_flat = X_pit_train.reshape(-1, n_feat)
X_pit_train_scaled = scaler_pit.fit_transform(X_pit_train_flat)
X_pit_train = X_pit_train_scaled.reshape(X_pit_train.shape)

X_pit_val_flat = X_pit_val.reshape(-1, n_feat)
X_pit_val_scaled = scaler_pit.transform(X_pit_val_flat)
X_pit_val = X_pit_val_scaled.reshape(X_pit_val.shape)

X_pit_test_flat = X_pit_test.reshape(-1, n_feat)
X_pit_test_scaled = scaler_pit.transform(X_pit_test_flat)
X_pit_test = X_pit_test_scaled.reshape(X_pit_test.shape)

In [ ]:
# Same for COMPOUND
X_comp_train_flat = X_comp_train.reshape(-1, n_feat)
X_comp_train_scaled = scaler_comp.fit_transform(X_comp_train_flat)
X_comp_train = X_comp_train_scaled.reshape(X_comp_train.shape)

X_comp_val_flat = X_comp_val.reshape(-1, n_feat)
X_comp_val_scaled = scaler_comp.transform(X_comp_val_flat)
X_comp_val = X_comp_val_scaled.reshape(X_comp_val.shape)

X_comp_test_flat = X_comp_test.reshape(-1, n_feat)
X_comp_test_scaled = scaler_comp.transform(X_comp_test_flat)
X_comp_test = X_comp_test_scaled.reshape(X_comp_test.shape)

### 4.4 Convert to PyTorch Tensors

In [ ]:
# PIT tensors
X_pit_train_t = torch.FloatTensor(X_pit_train)
y_pit_train_t = torch.FloatTensor(y_pit_train)
len_pit_train_t = torch.LongTensor(len_pit_train)

X_pit_val_t = torch.FloatTensor(X_pit_val)
y_pit_val_t = torch.FloatTensor(y_pit_val)
len_pit_val_t = torch.LongTensor(len_pit_val)

X_pit_test_t = torch.FloatTensor(X_pit_test)
y_pit_test_t = torch.FloatTensor(y_pit_test)
len_pit_test_t = torch.LongTensor(len_pit_test)

# COMPOUND tensors
X_comp_train_t = torch.FloatTensor(X_comp_train)
y_comp_train_t = torch.LongTensor(y_comp_train)
len_comp_train_t = torch.LongTensor(len_comp_train)

X_comp_val_t = torch.FloatTensor(X_comp_val)
y_comp_val_t = torch.LongTensor(y_comp_val)
len_comp_val_t = torch.LongTensor(len_comp_val)

X_comp_test_t = torch.FloatTensor(X_comp_test)
y_comp_test_t = torch.LongTensor(y_comp_test)
len_comp_test_t = torch.LongTensor(len_comp_test)

print("Tensors created ✓")

### 4.5 Class Weights

In [ ]:
# PIT class weights
pit_counts = np.bincount(y_pit_train.astype(int))
pit_total = len(y_pit_train)

pit_weight_0 = pit_total / (2 * pit_counts[0])
pit_weight_1 = pit_total / (2 * pit_counts[1])

pit_weights = torch.FloatTensor([pit_weight_0, pit_weight_1]).to(device)

print(f"PIT class weights:")
print(f"  No Pit (0): {pit_weight_0:.2f} (n={pit_counts[0]:,})")
print(f"  Pit (1):    {pit_weight_1:.2f} (n={pit_counts[1]:,})")

In [ ]:
# COMPOUND class weights
comp_counts = np.bincount(y_comp_train)
n_classes = len(comp_counts)
n_samples = len(y_comp_train)

comp_weight_dict = {}
for i in range(n_classes):
    comp_weight_dict[i] = n_samples / (n_classes * comp_counts[i])

comp_weights = torch.FloatTensor([comp_weight_dict[i] for i in range(n_classes)]).to(device)

print(f"\nCOMPOUND class weights:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls:12s}: {comp_weight_dict[i]:.2f} (n={comp_counts[i]:,})")

---
# 5. PyTorch Models

### 5.1 PIT Model Architecture

In [ ]:
class PitModel(nn.Module):
    """Binary classification for pit stop prediction"""
    
    def __init__(self, input_size, hidden_size=64, hidden_size2=32, dropout=0.4):
        super(PitModel, self).__init__()
        
        # Gaussian noise (applied in forward)
        self.noise_std = 0.05
        
        # LSTM layers
        self.lstm1 = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            dropout=dropout if dropout > 0 else 0
        )
        self.ln1 = nn.LayerNorm(hidden_size)
        
        self.lstm2 = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size2,
            batch_first=True,
            dropout=dropout if dropout > 0 else 0
        )
        
        # Dense layers
        self.fc1 = nn.Linear(hidden_size2, 48)
        self.bn1 = nn.BatchNorm1d(48)
        self.dropout1 = nn.Dropout(0.5)
        
        self.fc2 = nn.Linear(48, 24)
        self.dropout2 = nn.Dropout(0.3)
        
        self.fc_out = nn.Linear(24, 1)
        
    def forward(self, x, lengths=None):
        batch_size = x.size(0)
        
        # Add Gaussian noise during training
        if self.training:
            x = x + torch.randn_like(x) * self.noise_std
        
        # LSTM 1
        out, _ = self.lstm1(x)
        out = self.ln1(out)
        
        # LSTM 2
        out, (hidden, _) = self.lstm2(out)
        
        # Take last hidden state
        out = hidden[-1]  # (batch, hidden_size2)
        
        # Dense layers
        out = torch.relu(self.fc1(out))
        out = self.bn1(out)
        out = self.dropout1(out)
        
        out = torch.relu(self.fc2(out))
        out = self.dropout2(out)
        
        out = torch.sigmoid(self.fc_out(out))
        
        return out.squeeze(1)


# Initialize model
model_pit = PitModel(input_size=n_feat, hidden_size=64, hidden_size2=32, dropout=0.4)
model_pit = model_pit.to(device)

# Count parameters
pit_params = sum(p.numel() for p in model_pit.parameters())
print(f"PIT Model: {pit_params:,} parameters")
print(model_pit)

### 5.2 COMPOUND Model Architecture

In [ ]:
class CompoundModel(nn.Module):
    """Multi-class classification for compound prediction"""
    
    def __init__(self, input_size, num_classes, hidden_size=96, hidden_size2=48, 
                 num_heads=4, dropout=0.5):
        super(CompoundModel, self).__init__()
        
        self.noise_std = 0.05
        
        # LSTM 1
        self.lstm1 = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            dropout=dropout if dropout > 0 else 0
        )
        self.ln1 = nn.LayerNorm(hidden_size)
        
        # Multi-head attention
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True
        )
        self.ln2 = nn.LayerNorm(hidden_size)
        
        # LSTM 2
        self.lstm2 = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size2,
            batch_first=True,
            dropout=dropout if dropout > 0 else 0
        )
        
        # Dense layers
        self.fc1 = nn.Linear(hidden_size2, 48)
        self.bn1 = nn.BatchNorm1d(48)
        self.dropout1 = nn.Dropout(0.5)
        
        self.fc2 = nn.Linear(48, 24)
        self.dropout2 = nn.Dropout(0.4)
        
        self.fc_out = nn.Linear(24, num_classes)
        
    def forward(self, x, lengths=None):
        # Add noise during training
        if self.training:
            x = x + torch.randn_like(x) * self.noise_std
        
        # LSTM 1
        out, _ = self.lstm1(x)
        out = self.ln1(out)
        
        # Multi-head attention with residual
        attn_out, _ = self.attention(out, out, out)
        out = out + attn_out  # Residual connection
        out = self.ln2(out)
        
        # LSTM 2
        out, (hidden, _) = self.lstm2(out)
        out = hidden[-1]
        
        # Dense layers
        out = torch.relu(self.fc1(out))
        out = self.bn1(out)
        out = self.dropout1(out)
        
        out = torch.relu(self.fc2(out))
        out = self.dropout2(out)
        
        out = self.fc_out(out)  # Logits (softmax in loss)
        
        return out


# Initialize model
n_compound_classes = len(label_encoder.classes_)
model_comp = CompoundModel(
    input_size=n_feat, 
    num_classes=n_compound_classes,
    hidden_size=96, 
    hidden_size2=48,
    num_heads=4,
    dropout=0.5
)
model_comp = model_comp.to(device)

comp_params = sum(p.numel() for p in model_comp.parameters())
print(f"COMPOUND Model: {comp_params:,} parameters")
print(model_comp)

### 5.3 Training Setup

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    
    for X_batch, y_batch, len_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch, len_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
    
    return total_loss / len(dataloader)


def eval_epoch(model, dataloader, criterion, device):
    """Evaluate for one epoch"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch, len_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            outputs = model(X_batch, len_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item()
            
            all_preds.append(outputs.cpu())
            all_labels.append(y_batch.cpu())
    
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    
    return total_loss / len(dataloader), all_preds, all_labels


class EarlyStopping:
    """Early stopping to prevent overfitting"""
    
    def __init__(self, patience=15, min_delta=0.001, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0
        
    def __call__(self, score, epoch):
        if self.mode == 'min':
            score = -score
        
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_epoch = epoch
            self.counter = 0
        
        return self.early_stop

### 5.4 Train PIT Model

In [ ]:
# Create DataLoaders
train_dataset_pit = TensorDataset(X_pit_train_t, y_pit_train_t, len_pit_train_t)
val_dataset_pit = TensorDataset(X_pit_val_t, y_pit_val_t, len_pit_val_t)

train_loader_pit = DataLoader(train_dataset_pit, batch_size=128, shuffle=True)
val_loader_pit = DataLoader(val_dataset_pit, batch_size=128, shuffle=False)

In [ ]:
# Training setup
criterion_pit = nn.BCELoss(weight=None)  # Will use weighted loss per sample
optimizer_pit = optim.Adam(model_pit.parameters(), lr=0.002)
scheduler_pit = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_pit, mode='min', factor=0.5, patience=4, min_lr=1e-6
)
early_stopping_pit = EarlyStopping(patience=12, mode='min')

# Training loop
print("Training PIT model...")
print("="*60)

history_pit = {'train_loss': [], 'val_loss': [], 'val_auc': []}
best_val_loss = float('inf')
best_model_state = None

for epoch in range(50):
    train_loss = train_epoch(model_pit, train_loader_pit, criterion_pit, optimizer_pit, device)
    val_loss, val_preds, val_labels = eval_epoch(model_pit, val_loader_pit, criterion_pit, device)
    
    # Compute AUC
    val_auc = roc_auc_score(val_labels.numpy(), val_preds.numpy())
    
    history_pit['train_loss'].append(train_loss)
    history_pit['val_loss'].append(val_loss)
    history_pit['val_auc'].append(val_auc)
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model_pit.state_dict().copy()
    
    # Learning rate scheduling
    scheduler_pit.step(val_loss)
    
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f}")
    
    # Early stopping
    if early_stopping_pit(val_loss, epoch):
        print(f"Early stopping at epoch {epoch+1}")
        print(f"Best epoch: {early_stopping_pit.best_epoch+1}")
        break

# Restore best model
model_pit.load_state_dict(best_model_state)
print("\nTraining complete! Best model restored.")

### 5.5 Train COMPOUND Model

In [ ]:
# Create DataLoaders
train_dataset_comp = TensorDataset(X_comp_train_t, y_comp_train_t, len_comp_train_t)
val_dataset_comp = TensorDataset(X_comp_val_t, y_comp_val_t, len_comp_val_t)

train_loader_comp = DataLoader(train_dataset_comp, batch_size=256, shuffle=True)
val_loader_comp = DataLoader(val_dataset_comp, batch_size=256, shuffle=False)

In [ ]:
# Training setup
criterion_comp = nn.CrossEntropyLoss(weight=comp_weights)
optimizer_comp = optim.Adam(model_comp.parameters(), lr=0.0005)
scheduler_comp = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_comp, mode='max', factor=0.5, patience=6, min_lr=1e-6
)
early_stopping_comp = EarlyStopping(patience=15, mode='max')

# Training loop
print("\nTraining COMPOUND model...")
print("="*60)

history_comp = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_model_state_comp = None

for epoch in range(60):
    train_loss = train_epoch(model_comp, train_loader_comp, criterion_comp, optimizer_comp, device)
    val_loss, val_preds, val_labels = eval_epoch(model_comp, val_loader_comp, criterion_comp, device)
    
    # Compute accuracy
    val_preds_class = torch.argmax(val_preds, dim=1)
    val_acc = (val_preds_class == val_labels).float().mean().item()
    
    history_comp['train_loss'].append(train_loss)
    history_comp['val_loss'].append(val_loss)
    history_comp['val_acc'].append(val_acc)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state_comp = model_comp.state_dict().copy()
    
    # Learning rate scheduling
    scheduler_comp.step(val_acc)
    
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Early stopping
    if early_stopping_comp(val_acc, epoch):
        print(f"Early stopping at epoch {epoch+1}")
        print(f"Best epoch: {early_stopping_comp.best_epoch+1}")
        break

# Restore best model
model_comp.load_state_dict(best_model_state_comp)
print("\nTraining complete! Best model restored.")

---
# 6. Evaluation

### 6.1 PIT Model Evaluation

In [ ]:
# Test evaluation
test_dataset_pit = TensorDataset(X_pit_test_t, y_pit_test_t, len_pit_test_t)
test_loader_pit = DataLoader(test_dataset_pit, batch_size=128, shuffle=False)

model_pit.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch, len_batch in test_loader_pit:
        X_batch = X_batch.to(device)
        outputs = model_pit(X_batch, len_batch)
        all_preds.append(outputs.cpu())
        all_labels.append(y_batch)

y_pit_pred_prob = torch.cat(all_preds).numpy()
y_pit_true = torch.cat(all_labels).numpy()

# Find optimal threshold
prec, rec, thresh = precision_recall_curve(y_pit_true, y_pit_pred_prob)
f1_arr = 2 * prec * rec / (prec + rec + 1e-8)
best_idx = np.argmax(f1_arr)
opt_thresh = thresh[best_idx] if best_idx < len(thresh) else 0.5

y_pit_pred = (y_pit_pred_prob > opt_thresh).astype(int)

# Metrics
pit_auc = roc_auc_score(y_pit_true, y_pit_pred_prob)
pit_f1 = f1_score(y_pit_true, y_pit_pred)
pit_acc = accuracy_score(y_pit_true, y_pit_pred)
cm_pit = confusion_matrix(y_pit_true, y_pit_pred)

print("\n" + "-"*50)
print("PIT MODEL (PitIn3Laps)")
print("-"*50)
print(f"AUC-ROC:   {pit_auc:.3f}")
print(f"F1 Score:  {pit_f1:.3f}")
print(f"Accuracy:  {pit_acc:.3f}")
print(f"Precision: {prec[best_idx]:.3f}")
print(f"Recall:    {rec[best_idx]:.3f}")
print(f"Threshold: {opt_thresh:.3f}")
print(f"\nConfusion Matrix:")
print(f"           Pred:0  Pred:1")
print(f"True:0     {cm_pit[0,0]:6d}  {cm_pit[0,1]:6d}")
print(f"True:1     {cm_pit[1,0]:6d}  {cm_pit[1,1]:6d}")

### 6.2 COMPOUND Model Evaluation

In [ ]:
# Test evaluation
test_dataset_comp = TensorDataset(X_comp_test_t, y_comp_test_t, len_comp_test_t)
test_loader_comp = DataLoader(test_dataset_comp, batch_size=256, shuffle=False)

model_comp.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch, len_batch in test_loader_comp:
        X_batch = X_batch.to(device)
        outputs = model_comp(X_batch, len_batch)
        all_preds.append(outputs.cpu())
        all_labels.append(y_batch)

y_comp_pred_logits = torch.cat(all_preds)
y_comp_pred = torch.argmax(y_comp_pred_logits, dim=1).numpy()
y_comp_true = torch.cat(all_labels).numpy()

# Metrics
comp_acc = accuracy_score(y_comp_true, y_comp_pred)
comp_f1 = f1_score(y_comp_true, y_comp_pred, average='weighted')
cm_comp = confusion_matrix(y_comp_true, y_comp_pred)

print("\n" + "-"*50)
print("COMPOUND MODEL (NextCompound)")
print("-"*50)
print(f"Accuracy:  {comp_acc:.3f}")
print(f"F1 (weighted): {comp_f1:.3f}")

# Per-class accuracy
print(f"\nPer-class accuracy:")
for i, cls in enumerate(label_encoder.classes_):
    class_mask = (y_comp_true == i)
    if class_mask.sum() > 0:
        class_acc = (y_comp_pred[class_mask] == i).mean()
        n_true = class_mask.sum()
        n_pred = (y_comp_pred == i).sum()
        print(f"  {cls:12s}: {class_acc:.3f} (n={n_true:,}, pred={n_pred:,})")

print(f"\nConfusion Matrix:")
print(f"Classes: {list(label_encoder.classes_)}")
print(cm_comp)

### 6.3 Training Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# PIT - Loss
axes[0, 0].plot(history_pit['train_loss'], label='Train', linewidth=2)
axes[0, 0].plot(history_pit['val_loss'], label='Validation', linewidth=2)
axes[0, 0].set_title('PIT Model - Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# PIT - AUC
axes[0, 1].plot(history_pit['val_auc'], label='Validation', linewidth=2)
axes[0, 1].axhline(y=pit_auc, color='r', linestyle='--', label=f'Test AUC: {pit_auc:.3f}')
axes[0, 1].set_title('PIT Model - AUC', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# COMPOUND - Loss
axes[1, 0].plot(history_comp['train_loss'], label='Train', linewidth=2)
axes[1, 0].plot(history_comp['val_loss'], label='Validation', linewidth=2)
axes[1, 0].set_title('COMPOUND Model - Loss', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# COMPOUND - Accuracy
axes[1, 1].plot(history_comp['val_acc'], label='Validation', linewidth=2)
axes[1, 1].axhline(y=comp_acc, color='r', linestyle='--', label=f'Test Acc: {comp_acc:.3f}')
axes[1, 1].set_title('COMPOUND Model - Accuracy', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Other/training_history_pytorch.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 7. Save Models

In [ ]:
# Save PyTorch models
torch.save(model_pit.state_dict(), 'Model/f1_pit_model_pytorch.pt')
torch.save(model_comp.state_dict(), 'Model/f1_compound_model_pytorch.pt')

# Save full models (with architecture)
torch.save(model_pit, 'Model/f1_pit_model_pytorch_full.pt')
torch.save(model_comp, 'Model/f1_compound_model_pytorch_full.pt')

print("PyTorch models saved ✓")

In [ ]:
# Save scalers and encoder
joblib.dump(scaler_pit, 'Model/f1_pit_scaler_pytorch.pkl')
joblib.dump(scaler_comp, 'Model/f1_comp_scaler_pytorch.pkl')
joblib.dump(label_encoder, 'Model/label_encoder_pytorch.pkl')

print("Scalers saved ✓")

In [ ]:
# Save configuration
config = {
    'sequence_length': SEQUENCE_LENGTH,
    'FEATURES': FEATURES,
    'pit_threshold': float(opt_thresh),
    'compound_classes': list(label_encoder.classes_),
    'metrics': {
        'pit_auc': float(pit_auc),
        'pit_f1': float(pit_f1),
        'pit_accuracy': float(pit_acc),
        'compound_accuracy': float(comp_acc),
        'compound_f1': float(comp_f1)
    },
    'framework': 'pytorch',
    'device': str(device)
}

with open('Model/modelConfig_pytorch.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Config saved ✓")

In [ ]:
# Save dataset with features
df_f1.to_pickle('Other/f1_dataset_featured_pytorch.pkl')
print("Dataset saved ✓")